https://github.com/facebookresearch/faiss/wiki/Running-on-GPUs

In [ ]:
from huggingface_hub import hf_hub_download
organization_dataset_id="hatemestinbejaia/STCALIR_Synthetic-Test-Collection"
hf_hub_download(repo_id=organization_dataset_id, filename="Corpus Algerian Legal Texts.csv", local_dir="./", repo_type="dataset")
import pandas
df = pandas.read_csv("Corpus Algerian Legal Texts.csv")
min_len = df['word_count'].min()
max_len = df['word_count'].max()
mean_len = df['word_count'].mean()
std_len = df['word_count'].std()

print(f"Min text length: {min_len}")
print(f"Max text length: {max_len}")
print(f"Mean text length: {mean_len:.2f}")
print(f"Std text length: {std_len:.2f}")
df = df[['passage_id', 'text_with_summary']]
# Assume your DataFrame is df_passages
df = df.rename(columns={'passage_id': 'id', 'text_with_summary': 'text'})
df["id"] = df["id"].replace(' ','_', regex=True)
# Optional: reset the index
df = df.reset_index(drop=True)
df.to_csv('STCALIR_collection.tsv', sep="\t", header=None,index=False) 
print(len(df))
df.head(1)

In [ ]:
from huggingface_hub import hf_hub_download
organization_dataset_id="hatemestinbejaia/STCALIR_Synthetic-Test-Collection"
hf_hub_download(repo_id=organization_dataset_id, filename="Topics Algerian Legal Texts.csv", local_dir="./", repo_type="dataset")
df_topic = pandas.read_csv("Topics Algerian Legal Texts.csv")
df_topic = df_topic.reset_index()[["topic_id", "topic_title"]]
df_topic = df_topic.rename(columns={'topic_id': 'id', 'topic_title': 'text'})
df_topic.to_csv('STCALIR-Topics.tsv', sep="\t", header=None,index=False) 
print(df_topic.head())
# Compute text length (in words)
df_topic['text_length'] = df_topic['text'].apply(lambda x: len(str(x).split()))

# Compute statistics
min_len = df_topic['text_length'].min()
max_len = df_topic['text_length'].max()
mean_len = df_topic['text_length'].mean()
std_len = df_topic['text_length'].std()

print(f"Min text length: {min_len}")
print(f"Max text length: {max_len}")
print(f"Mean text length: {mean_len:.2f}")
print(f"Std text length: {std_len:.2f}")

In [ ]:
# pyserini love java (install java)
!sudo apt-get update
!sudo apt-get install openjdk-21-jre -y
!java -version
!sudo  update-java-alternatives --list
!export JAVA_HOME='/usr/lib/jvm/java-1.21.0-openjdk-amd64'
import os
# linux
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-1.21.0-openjdk-amd64"
# installing pyserini
#https://github.com/castorini/pyserini
!python --version
!pip install faiss-cpu -q
!pip install pyserini==0.22.0 -q
#!pip install transformers==4.36.2 -q

# install numpy==1.26.4 to fix package conflit 
import numpy 
numpy.__version__
#!pip show numpy
#!pip install --upgrade numpy==1.26.4

In [ ]:
# clone mMarco code
!git clone https://github.com/unicamp-dl/mMARCO.git

In [ ]:
import gc
del df
gc.collect()
!git clone --recursive https://github.com/castorini/pygaggle.git

In [ ]:
!python pygaggle/tools/scripts/msmarco/convert_collection_to_jsonl.py \
    --collection-path STCALIR_collection.tsv\
    --output-folder collections/STCALIR-passage/collection_jsonl

In [ ]:
!python -m pyserini.index.lucene \
  --collection JsonCollection \
  --input collections/STCALIR-passage/collection_jsonl/ \
  --index indexes/STCALIR-lucene-index \
  --generator DefaultLuceneDocumentGenerator \
  --threads 1 \
  --storePositions --storeDocvectors --storeRaw \
  --language ar

In [ ]:
!python -m pyserini.search.lucene \
  --index indexes/STCALIR-lucene-index \
  --topics STCALIR-Topics.tsv \
  --output runs/run.STCALIR.bm25tuned.txt \
  --output-format msmarco \
  --hits 1000 \
  --bm25 --k1 0.82 --b 0.68 \
  --language ar

In [ ]:
!pip install sentence_transformers -q

In [ ]:
from sentence_transformers import SentenceTransformer, util
organization_dataset_id="hatemestinbejaia/STCALIR_Synthetic-Test-Collection"
collection_name = 'STCALIR_collection.tsv'
queries_name = 'STCALIR-Topics.tsv'
list_of_models= [ "hatemestinbejaia/mmarco-Arabic-AraElectra-bi-encoder-NoKD-v1", 
              "hatemestinbejaia/mmarco-Arabic-AraElectra-bi-encoder-KD-v1",
              "hatemestinbejaia/mmarco-Arabic-AraDPR-bi-encoder-NoKD-v1",
              "hatemestinbejaia/mmarco-Arabic-AraDPR-bi-encoder-KD-v1",
              "hatemestinbejaia/mmarco-Arabic-mMiniLML-bi-encoder-NoKD-v1",
              "hatemestinbejaia/mmarco-Arabic-mMiniLML-bi-encoder-KD-v1",
            ]

def embedding(batch):
    with torch.no_grad(): batch["embedding"] = model.encode(batch["text"], convert_to_tensor=True, show_progress_bar=False, batch_size=128)
    return batch

In [ ]:
from huggingface_hub import login
access_token = "your huggingface apikey"
# This saves your token for use by Transformers, pipelines, etc.
login(token=access_token)

In [ ]:
df = pandas.read_csv(collection_name, sep="\t", header=None, names=["id", "text"])
from datasets import Dataset
import torch
for i in list_of_models:
    # Convert to Hugging Face Dataset
    datasetC = Dataset.from_pandas(df)
    print(datasetC)
    model = SentenceTransformer(i)
    #print(model)
    model.eval()
    datasetC= datasetC.map(embedding, batched=True)
    print(datasetC)
    datasetC.push_to_hub(organization_dataset_id, "STCALIR_collection_embedding_"+i.split("/")[1])
    #break


In [ ]:
df = pandas.read_csv(queries_name, sep="\t", header=None, names=["id", "text"])
# Convert to Hugging Face Dataset
from datasets import Dataset, DatasetDict
import torch
for i in list_of_models:
    datasetQ = Dataset.from_pandas(df)
    datasetQ = DatasetDict({"train": datasetQ})
    model = SentenceTransformer(i)
    #print(model)
    model.eval()
    datasetQ= datasetQ.map(embedding, batched=True)
    datasetQ.push_to_hub(organization_dataset_id, "STCALIR-Topics_embedding_"+i.split("/")[1])
    #break

In [ ]:
import numpy as np
import gc
from datasets import load_dataset
import huggingface_hub
import pyarrow
import datasets
import csv
import os
import pickle
import time
import faiss
from tqdm import trange 
# embedding vector generated with our bi-encoder model
organization_dataset_id="hatemestinbejaia/STCALIR_Synthetic-Test-Collection"
mmarco_embedding_collection_list =["STCALIR_collection_embedding_mmarco-Arabic-AraDPR-bi-encoder-KD-v1", "STCALIR_collection_embedding_mmarco-Arabic-AraDPR-bi-encoder-NoKD-v1", "STCALIR_collection_embedding_mmarco-Arabic-AraElectra-bi-encoder-KD-v1","STCALIR_collection_embedding_mmarco-Arabic-AraElectra-bi-encoder-NoKD-v1","STCALIR_collection_embedding_mmarco-Arabic-mMiniLML-bi-encoder-KD-v1","STCALIR_collection_embedding_mmarco-Arabic-mMiniLML-bi-encoder-NoKD-v1", ]
mmarco_embedding_queries_list=["STCALIR-Topics_embedding_mmarco-Arabic-AraDPR-bi-encoder-KD-v1", "STCALIR-Topics_embedding_mmarco-Arabic-AraDPR-bi-encoder-NoKD-v1", "STCALIR-Topics_embedding_mmarco-Arabic-AraElectra-bi-encoder-KD-v1","STCALIR-Topics_embedding_mmarco-Arabic-AraElectra-bi-encoder-NoKD-v1","STCALIR-Topics_embedding_mmarco-Arabic-mMiniLML-bi-encoder-KD-v1","STCALIR-Topics_embedding_mmarco-Arabic-mMiniLML-bi-encoder-NoKD-v1", ]
#print("PyArrow version:", pyarrow.__version__)
#print("Datasets version:", datasets.__version__)

In [ ]:
def load_embedding(embedding):
    print(embedding)
    global output_run
    output_run+= mmarco_embedding_collection_list[embedding]
    # load corpus embedding
    corpus_embeddings = load_dataset(organization_dataset_id, mmarco_embedding_collection_list[embedding])
    print(corpus_embeddings) 
    corpus_embeddings=corpus_embeddings["train"].with_format("np")
    #corpus_embeddings=corpus_embeddings.map(lambda x: {"id": x["id"].replace(" ", "_")})
    corpus_id=corpus_embeddings["id"]
    corpus_embeddings=corpus_embeddings["embedding"]
    corpus_embeddings
    # Force a garbage collection
    gc.collect()
    # load queries embedding
    queries_embeddings = load_dataset(organization_dataset_id, mmarco_embedding_queries_list[embedding], trust_remote_code=True)
    print(queries_embeddings) 
    queries_embeddings = queries_embeddings["train"].with_format("np")
    queries_id=queries_embeddings["id"]
    queries_embeddings=queries_embeddings["embedding"]
    #print(queries_embeddings)
    print(len(queries_embeddings[0]))
    return corpus_embeddings,corpus_id, queries_embeddings, queries_id

In [ ]:
#Approximate Nearest Neighbor Search (ANN) with FAISS (https://github.com/facebookresearch/faiss).
def creat_ANN_faiss_index(corpus_embeddings,corpus_id, queries_embeddings, queries_id):
    global output_run
    embedding_size = len(corpus_embeddings[0])  # Size of embeddings
    print(embedding_size)
    ### Create the FAISS index
    print("Start creating FAISS index")
    index = faiss.index_factory(embedding_size, "Flat", faiss.METRIC_INNER_PRODUCT)
    corpus_embeddings = np.stack(corpus_embeddings, axis=0).astype('float32')
    faiss.normalize_L2(corpus_embeddings)
    print("end norm")
    # Then we train the index to find a suitable clustering
    index.train(corpus_embeddings)
    print("end train")
    faiss.write_index(index,"tmp.index")
    index = faiss.read_index("tmp.index")
    index.add(corpus_embeddings)
    faiss.write_index(index, "tmp.index")
    print("faiss index built it ok")
    print("total index:",index.ntotal)
    # free some RAM
    del corpus_embeddings
    # Force a garbage collection
    gc.collect()
    output_run += "_top_k_hits_1000.tsv"
    print(output_run)

In [ ]:
######### Search in the index ###########
def search_the_index():
    global output_run, corpus_embeddings,corpus_id, queries_embeddings, queries_id
    index = faiss.read_index("tmp.index")
    top_k_hits=1000
    with open(output_run, 'w', encoding='utf-8') as f_out:
        for i in trange(len(queries_embeddings)):
            start_time = time.time()
            question_embedding = queries_embeddings[i]
            # FAISS works with inner product (dot product). When we normalize vectors to unit length, inner product is equal to cosine similarity
            #question_embedding = question_embedding / np.linalg.norm(question_embedding)
            question_embedding = np.expand_dims(question_embedding, axis=0)
            faiss.normalize_L2(question_embedding)
    
            # Search in FAISS. It returns a matrix with distances and corpus ids.
            distances, corpus_ids = index.search(question_embedding, top_k_hits)
    
            # We extract corpus ids and scores for the first query
            hits = [{"corpus_id": id, "score": score} for id, score in zip(corpus_ids[0], distances[0])]
            #print(hits)
            hits = sorted(hits, key=lambda x: x["score"], reverse=True)
            end_time = time.time()
            for j in range(len(hits)):
                f_out.write(f'{queries_id[i]}\t{corpus_id[hits[j]["corpus_id"]]}\t{j+1}\n')    
            #break

In [ ]:
for i in range(len(mmarco_embedding_collection_list)):
    output_run="STCALIR_first_stage_"
    corpus_embeddings,corpus_id, queries_embeddings, queries_id=load_embedding(i)
    creat_ANN_faiss_index(corpus_embeddings,corpus_id, queries_embeddings, queries_id)
    search_the_index()
    print(i)
    #break

In [ ]:
import os
prefix = "STCALIR_first_stage_"
files = [f for f in os.listdir(".") if f.startswith(prefix)]
files.append("runs/run.STCALIR.bm25tuned.txt")
print(files)

In [ ]:
import pandas as pd
from collections import defaultdict
import glob

def load_ranking_file(path):
    """Load a ranking file of format: qid, docid, rank."""
    return pd.read_csv(path, sep="\t", names=["qid", "docid", "rank"])

def rrf(ranking_lists, k=60):
    """Apply RRF fusion to a list of ranking lists (docid lists)."""
    scores = defaultdict(float)
    for ranking in ranking_lists:
        for rank, docid in enumerate(ranking):
            scores[docid] += 1.0 / (k + rank + 1)
    # Sort documents by RRF score
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def rrf_fusion(run_files, output_path=None, k=60, top_k=1000):
    """
    Apply RRF to multiple ranking files with identical structure.
    
    run_files: list of file paths
    output_path: where to save fused run (optional)
    Returns: fused DataFrame
    """
    
    # 1) Load all runs
    runs = [load_ranking_file(f) for f in run_files]

    # All topics from first file
    topics = runs[0]["qid"].unique()
    #topics = sorted(set().union(*[df["qid"].unique() for df in runs]))


    fused_rows = []

    for qid in topics:
        ranking_lists = []

        # 2) Extract docids per topic from each run
        for df in runs:
            docs = df[df["qid"] == qid].sort_values("rank")["docid"].tolist()
            ranking_lists.append(docs)

        # 3) Apply RRF
        fused_docs = rrf(ranking_lists, k=k)

        # 4) Keep top_k
        for rank, (docid, score) in enumerate(fused_docs[:top_k], start=1):
            fused_rows.append([qid, docid, rank])

    # Create DataFrame
    fused_df = pd.DataFrame(fused_rows, columns=["qid", "docid", "rank"])

    # Optional save
    if output_path:
        fused_df.to_csv(output_path, sep="\t", index=False, header=False)
        print(f"Saved RRF run to: {output_path}")

    return fused_df

In [ ]:
rrf_fusion(files, output_path="STCALIR_bm25_6faiss_rrf.txt")

In [ ]:
api.upload_file(
    path_or_fileobj="STCALIR_bm25_6faiss_rrf.txt",
    path_in_repo="STCALIR_bm25_6faiss_rrf.txt",
    repo_id=organization_dataset_id,
    repo_type="dataset",
    )

!pip install -U datasets huggingface-hub -q
import huggingface_hub 
hf = huggingface_hub.HfFolder()
# to load the reranked list to your Hugging Face repository, enter your access token below. 
# and change the repository ID to yours.
#organization_dataset_id =""
access_token = "your huggingface apikey"
hf.save_token(access_token)
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(path_or_fileobj=output_run, path_in_repo=output_run, repo_id=organization_dataset_id, repo_type="dataset",)